# Lab 14 — A learned model on spectrograms vs a well-featured forest — fairly

> **Type-2 lab — student version.** Fill in each `# TODO` (the answer cells raise `NotImplementedError` until you do), answer the **Checkpoint** questions, and complete the reflection cell, then re-run top to bottom. The instructor solution lives in `labs/solutions/` and is not included here.

**Covers.** Chapter 14 — §14.4 (CNNs), §14.7 (practical reality: a fair fight with the forest).

**Biomedical question.** Does a model that *learns* from the raw time–frequency image beat a hand-featured forest here — and is the comparison **fair**?
**Task type (§1.8).** Deep-vs-classical classification — an honest, like-for-like comparison.
**Information that must be preserved.** *two* things at once — **fairness** (the SAME subjects held out on the SAME folds for both models) **and** **honesty** (never a resubstitution / in-sample number reported as skill, never an unfair claim).
**Main assumptions.** the class cue is a time–frequency **burst**; every epoch of a subject carries that subject's fixed nuisance tone, so subjects stay whole across the split (GroupKFold).
**Primary diagnostic.** subject-independent Cohen's kappa on shared folds vs the majority-class baseline; a **data-efficiency** sweep; and the resubstitution **optimism gap**.
**Transfer challenge.** does the verdict — and the data-efficiency trend — survive with many more subjects, a new site, or a real weight-sharing CNN?

*Self-contained: a seeded synthetic multi-subject cohort, no data files, no `bsp`. Runs fully offline in well under a minute. Theme (§1.8): there is often **no single best** method — and on a **small** set the well-featured forest frequently **ties or wins**, because the learned model is data-hungry.*

### Companion lab · *Biomedical Signal Processing & Data Analytics* (CM2013)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/labs/lab14_cnn_spectrogram_vs_forest/lab14_cnn_spectrogram_vs_forest.ipynb) [![nbviewer](https://img.shields.io/badge/view-nbviewer-orange)](https://nbviewer.org/github/farhad-abtahi/CM2013/blob/main/labs/lab14_cnn_spectrogram_vs_forest/lab14_cnn_spectrogram_vs_forest.ipynb) [![JupyterLite](https://img.shields.io/badge/run-JupyterLite-blue)](https://farhad-abtahi.github.io/CM2013/lite/lab/index.html?path=lab14_cnn_spectrogram_vs_forest.ipynb)

In [ ]:
# --- shared setup (reproducible; offline synthetic multi-subject cohort) ---
import warnings
import numpy as np, matplotlib.pyplot as plt
from scipy import signal as sig
from scipy.ndimage import zoom
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, GroupKFold
from sklearn.metrics import cohen_kappa_score, accuracy_score
warnings.filterwarnings('ignore')          # keep MLP convergence chatter out of the lesson
rng = np.random.default_rng(2013)
plt.rcParams.update({'figure.dpi': 120, 'axes.grid': False})

FS   = 128           # Hz
SECS = 2.0
N    = int(FS * SECS)                      # 256 samples per epoch
CLASSES    = ['A', 'B', 'C']               # 3 EEG-like states
CLASS_FREQ = [12.0, 20.0, 30.0]            # burst carrier per class (Hz)
CLASS_TIME = [0.5, 1.0, 1.5]               # burst centre time per class (s) -> a MOVING T-F blob
BANDS      = [(8, 16), (16, 24), (24, 32), (32, 40)]   # feature bands (aligned to the carriers)
IMG  = 16                                  # spectrogram downsampled to IMG x IMG

def spectrogram_image(x):
    '''Small log-spectrogram image (IMG x IMG) -- the input the 'learned' model sees.'''
    f, tt, Sxx = sig.spectrogram(x, fs=FS, nperseg=32, noverlap=16)
    logS = np.log(Sxx + 1e-12)
    return zoom(logS, (IMG / logS.shape[0], IMG / logS.shape[1]), order=1)

def bandpower_features(x):
    '''Hand-crafted band-power features: 4 bands x 3 time-thirds = 12 numbers.'''
    feats = []
    for idx in np.array_split(np.arange(N), 3):            # early / mid / late third
        seg = x[idx]
        fr  = np.fft.rfftfreq(seg.size, d=1 / FS)
        P   = np.abs(np.fft.rfft(seg)) ** 2
        for lo, hi in BANDS:
            feats.append(np.log(P[(fr >= lo) & (fr < hi)].sum() + 1e-12))
    return np.array(feats)

def make_cohort(n_subj=9, per_class=20, noise=0.9, burst=1.0, nuis=0.8, seed=2013):
    '''Synthetic multi-subject cohort with a deliberate structure:
      * the CLASS cue is a time-frequency BURST -- carrier freq AND centre time differ by class,
        so the discriminative pattern is a blob that MOVES across the T-F image;
      * each SUBJECT adds a fixed NUISANCE tone at a subject-specific frequency, present in every
        one of that subject's epochs (a physiological / electrode idiosyncrasy). A learner can
        key on it in-sample, but an UNSEEN subject has a different tone -> whole subjects must be
        held out (GroupKFold), and a random-row split would flatter every model.
    Returns: flattened log-spectrogram images, band-power features, labels, subject groups.'''
    r = np.random.default_rng(seed)
    t = np.arange(N) / FS
    Ximg, Xfeat, y, g = [], [], [], []
    for s in range(n_subj):
        f_nuis   = r.uniform(9.0, 39.0)                    # this subject's private tone (the trap)
        phi_nuis = r.uniform(0, 2 * np.pi)
        for c in range(len(CLASSES)):
            for _ in range(per_class):
                blob = (burst * np.exp(-0.5 * ((t - CLASS_TIME[c]) / 0.12) ** 2)
                              * np.sin(2 * np.pi * CLASS_FREQ[c] * t + r.uniform(0, 2 * np.pi)))
                tone = nuis * np.sin(2 * np.pi * f_nuis * t + phi_nuis)
                x = blob + tone + noise * r.standard_normal(N)
                Ximg.append(spectrogram_image(x).ravel())
                Xfeat.append(bandpower_features(x))
                y.append(c); g.append(s)
    return np.array(Ximg), np.array(Xfeat), np.array(y), np.array(g)

In [ ]:
Ximg, Xfeat, y, groups = make_cohort()
print(f'cohort: {Ximg.shape[0]} epochs, {len(np.unique(groups))} subjects, classes = {CLASSES}')
print(f'  spectrogram image  : {Ximg.shape[1]} px  (= {IMG}x{IMG} flattened, the learned input)')
print(f'  band-power features: {Xfeat.shape[1]}     (4 bands x 3 time-thirds, the forest input)')
print('  per-class counts   :', np.bincount(y))

# ONE set of subject-grouped folds, built ONCE and reused by BOTH models -> the fairness rule.
cv = GroupKFold(n_splits=5)
splits = list(cv.split(Ximg, y, groups))    # identical folds handed to every model below
print(f'  folds: {len(splits)} subject-grouped folds (whole subjects held out, shared by both models)')

# peek: the class-mean log-spectrogram -- the moving T-F blob the learned model must find
fig, ax = plt.subplots(1, 3, figsize=(7.5, 2.5))
for c in range(len(CLASSES)):
    ax[c].imshow(Ximg[y == c].mean(0).reshape(IMG, IMG), origin='lower', aspect='auto')
    ax[c].set_title(f'class {CLASSES[c]}'); ax[c].set_xlabel('time ->'); ax[c].set_ylabel('freq ->')
fig.suptitle('class-mean log-spectrogram (the time-frequency cue)'); fig.tight_layout(); plt.show()

## 1. The 'learned' model — an MLP on the raw spectrogram image

The learned side gets **no hand features**: it sees only the flattened `16x16` log-spectrogram and must discover the class cue itself. To stay fast we use a small `MLPClassifier` as a **stand-in for a CNN-on-spectrogram**. (A real CNN would *share* weights spatially across the image instead of giving every pixel its own weight, so it needs fewer examples — but the lesson here, *learning from the raw T–F image vs from hand features*, is identical.) Score it with `cross_val_predict` on the **shared** subject-grouped folds; **Cohen's kappa**.

In [ ]:
# TODO train the 'learned' model: a small MLPClassifier on the FLATTENED log-spectrogram image.
#   (This stands in for a CNN-on-spectrogram; a real CNN SHARES weights spatially instead of
#    giving every pixel its own weight, but the lesson -- learning from the raw T-F image vs from
#    hand features -- is the same.) Put the StandardScaler INSIDE a pipeline so no fold leaks,
#    score with cross_val_predict on the SHARED `splits`, and report Cohen's kappa.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: 256 pixel-weights vs 12 hand features -- which model has more chances to overfit a
# small cohort? Why must the StandardScaler live INSIDE the pipeline, not be fit on all rows first?

## 2. The forest — a RandomForest on hand-crafted band-power features

The classical side gets only **12 numbers a human chose**: band power in 4 bands x 3 time-thirds — a compact encoding of *which band lights up when*, which is exactly the moving-blob cue. Train a `RandomForestClassifier` on those features, scored on the **identical** `splits`. Same folds is the whole point: it is the only way the two scores are comparable.

In [ ]:
# TODO train the forest: a RandomForestClassifier on the 12 band-power FEATURES, scored on the
#   SAME `splits` (identical subject-grouped folds) with Cohen's kappa. Same folds = fair race.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: the forest sees only 12 numbers; the MLP sees all 256 pixels. On THIS small cohort,
# which is ahead -- and is the gap bigger than fold-to-fold wobble?

## 3. The fair verdict — and data efficiency

Both kappas were computed on the **same folds**, so they can be read side by side against the **majority-class baseline** (kappa ≈ 0). Then the key question for a *learned* model: **how much data does it need?** Sweep the number of training subjects `k = 2, 4, 6`, testing on the held-out subjects, and watch the learned MLP climb from data-starved toward the steady forest — the **data-efficiency** signature that decides who wins on a small set.

In [ ]:
# TODO (a) majority-class baseline kappa (~0); confirm BOTH models beat it on the shared folds;
#      (b) the fair headline: both kappas side by side on identical folds, and the margin;
#      (c) a data-efficiency sweep: train on k = 2, 4, 6 subjects, test on the held-out subjects,
#          averaged over a few random subject draws -> show the learned MLP is data-HUNGRY (weak
#          with few subjects, climbing as subjects are added) while the featured forest is
#          STEADIER. Quote the crossover / gap-closing behaviour and state the honest headline.
raise NotImplementedError("TODO: implement this — see the comment above")
# Checkpoint: state the honest headline in one line. Does the learned model's disadvantage look
# permanent, or like a symptom of too few subjects? What single change would most likely flip it?

## 4. Live sanity check — fairness and honesty

One cell ties the lab together, every number computed live: **(1)** both models clear the majority-class baseline; **(2)** they were scored on the **SAME** folds and every fold holds whole subjects out (no subject leaks across a fold); **(3)** the **optimism gap** — for the *same* forest, resubstitution (fit and score the same rows) sits far above the subject-independent score. That gap is the in-sample optimism that honesty forbids reporting as skill.

In [ ]:
# --- live sanity check: fairness + honesty (all numbers computed live, none hard-coded) ---
# (1) BOTH models clear the majority-class baseline...
assert mlp_kappa > maj_kappa + 0.05, 'MLP does not beat the baseline'
assert rf_kappa  > maj_kappa + 0.05, 'RF does not beat the baseline'
# (2) ...on the SAME folds: the identical `splits` object drove both, and every fold holds whole
#     subjects out (train and test subjects disjoint) -> the comparison is fair.
for tr, te in splits:
    assert set(groups[tr]).isdisjoint(set(groups[te])), 'a subject leaked across a fold'
assert len(splits) == 5
# (3) the optimism gap: for the SAME forest, resubstitution must sit far ABOVE subject-independent.
rf_resub_kappa = cohen_kappa_score(
    y, RandomForestClassifier(n_estimators=300, random_state=0).fit(Xfeat, y).predict(Xfeat))
print(f'forest resubstitution kappa      = {rf_resub_kappa:.3f}   (memorised rows -- NOT skill)')
print(f'forest subject-independent kappa = {rf_kappa:.3f}   (unseen subjects -- the honest number)')
print(f'optimism gap                     = {rf_resub_kappa - rf_kappa:.3f}')
assert rf_resub_kappa > rf_kappa + 0.2, 'expected resubstitution >> subject-independent'
print('sanity check PASSED: both beat the baseline on the SAME folds; resubstitution >> grouped.')

## Reflection

This reflection is for your own practice — there is nothing to submit. What matters is the *reasoning*, not which model came out nominally first.

1. **Data-hungry or just worse?** From the §3 sweep, is the spectrogram-MLP's deficit a permanent quality gap or a symptom of too few subjects? Quote the `RF − MLP` gap at `k=2` vs `k=6` and say which way a real weight-sharing CNN would move it, and why.
2. **Winner or tie?** State your headline: on this small cohort, is there a single best model, or a tie/forest-win above the baseline? Cite the two grouped kappas (same folds) and the margin as evidence — and say what would have to change for the learned model to win.
3. **New site / device.** Which axis of the split changes when you move to a new recording site, and what would you re-measure before claiming either model transfers — for BOTH the spectrogram input and the hand features?

**Rule out (name the unfair comparison).** Give one concrete *wrong* way to run this study — e.g. scoring the MLP and the forest on **different** splits (each on its own lucky fold), or reporting the §4 **resubstitution / in-sample** score as ‘the CNN wins’ — and name the requirement it breaks: it violates the **§1.8 worldview** that a score must (a) represent the *claim* (skill on an unseen subject) and (b) be *fair* (the SAME folds for every model). Tie it to the theme: here there is **no single best** method — the forest ties or wins on this small set — **but wrong choices still exist**, and an unfair or in-sample comparison is one of them.

> *Your answers here.*

---
*Type-2 lab for **Biomedical Signal Processing & Data Analytics**. Synthetic cohort; illustrative numbers. The lesson is the method, not the winner: compare fairly (same folds), report against a baseline, weigh data efficiency, and claim only what the split can support.*